# SEC Filings Pipeline

Full pipeline from scraping 10-K filings through BigQuery AI extraction and property graph creation.

## Pipeline Steps
1. **Google Cloud authentication** – gcloud auth
2. **Scraper** – Download 10-K filings from SEC.gov
3. **Parser** – Parse SGML to Markdown (optional, for viewing)
4. **Extract Sections** – Parse SGML to JSONL (Item 1, 1A, 7; feeds BigQuery)
5. **Upload JSONL to GCS** – Sync JSON to BigQuery load bucket
6. **Init Tables** – Create sections, insights, staging tables
7. **Extraction** – AI extraction (Gemini) → insights
8. **Create Graph** – Transform insights to graph_edges
9. **Prepare Property Graph** – Node and edge tables
10. **Add Action to Market** – market_action column
11. **Create Property Graph DDL** – SecGraph property graph

### 0.0 GCP Environment Setup Checklist
Before running this notebook, ensure your Google Cloud environment is fully configured:

1.  **API Enablement**: Enable [BigQuery API](https://console.cloud.google.com/marketplace/product/google/bigquery.googleapis.com) and [Vertex AI API](https://console.cloud.google.com/marketplace/product/google/aiplatform.googleapis.com).
2.  **Billing**: Ensure your project is linked to an active **[Billing Account](https://console.cloud.google.com/billing/enable)**. (Note: Propagation can take 5-10 minutes).
3.  **BigQuery AI Connection**:
    - Create a Cloud Resource Connection named **`vertex_ai_connection`** in the **US** location.
    - Grant the connection's Service Account the **`roles/aiplatform.user`** (Vertex AI User) role.
    - See [documentation](https://cloud.google.com/bigquery/docs/generate-text#create_connection) for CLI commands.
4.  **Local Isolation**: If running in a Conda environment with other projects, this notebook uses `PYTHONPATH=""` for `bq` commands to avoid namespace conflicts.

---



## Configuration

Set the environment constants below. These variables control GCP authentication, local and cloud storage, and pipeline depth.

### GCP Settings
- **`GCP_PROJECT`**: Your Google Cloud Project ID. Find this in the [GCP Console Project Dashboard](https://console.cloud.google.com/home/dashboard). Ensure you have the [BigQuery API](https://console.cloud.google.com/marketplace/product/google/bigquery.googleapis.com) and [Vertex AI API](https://console.cloud.google.com/marketplace/product/google/aiplatform.googleapis.com) enabled.
- **`GCS_BUCKET`**: The Google Cloud Storage bucket used for staging data for BigQuery load. Format: `gs://your-bucket-name`.
- **`BQ_LOCATION`**: The regional location for your BigQuery datasets (e.g., `US`, `EU`).
- **`GEMINI_MODEL`**: The Gemini model version used for text extraction (e.g., `gemini-3.5-pro`).
- **Billing**: BigQuery AI functions require an active billing account. Ensure it is enabled [here](https://console.cloud.google.com/billing/enable).

### Local Directories
These variables define where data is stored on your local machine or external drive.
- **`SGML_DIR`**: Root directory for raw `.sgml` files from SEC.gov.
- **`MARKDOWN_DIR`**: Directory for converted `.md` files.
- **`JSON_DIR`**: Directory for sectioned JSONL files (Item 1, 1A, 7).

### Scraper Settings
- **`SCRAPER_LIMIT`**: The number of companies to pull from the companies list (e.g., 5, 20, 500). Use a small number to test.
- **`SCRAPER_LAST_N_YEARS`**: The number of historical years to scrape for each company.

In [ ]:
# Configure these for your environment
import os

GCP_PROJECT = "YOUR_PROJECT_ID"  # Change to your project ID
GCS_BUCKET = "gs://YOUR_BUCKET"  # Change to your bucket
BQ_LOCATION = "US"
GEMINI_MODEL = "gemini-3.1-pro-preview"  # Vertex AI model endpoint

# Local data directories
SGML_DIR = "./data/sgml"
MARKDOWN_DIR = "./data/markdown"
JSON_DIR = "./data/json"

# Pipeline depth
SCRAPER_LIMIT = 5           # Companies from list.csv (use a small number for testing, the max is 500)
SCRAPER_LAST_N_YEARS = 2      # Years of filings to fetch
SCRAPER_OUTPUT = SGML_DIR

# Derived project constants
BQ_PROJECT = GCP_PROJECT

# Assert configuration is updated
assert GCP_PROJECT != "YOUR_PROJECT_ID", "Update GCP_PROJECT at the top of the notebook."
assert GCS_BUCKET != "gs://YOUR_BUCKET", "Update GCS_BUCKET at the top of the notebook."

# Create local data directories if they don't exist
for d in [SGML_DIR, MARKDOWN_DIR, JSON_DIR]:
    os.makedirs(d, exist_ok=True)
print(f"Verified directories: {SGML_DIR}, {MARKDOWN_DIR}, {JSON_DIR}")


## 0. Colab Setup
If you are running this in Google Colab, you need to clone the repository and install requirements to access the custom scraper scripts and SQL files.

In [ ]:
import sys
import os
from IPython import get_ipython
ipy = get_ipython()
if 'google.colab' in sys.modules:
    if not os.path.exists('00.0_scraper.py'):
        ipy.system('git clone https://github.com/Kineviz/fortune500.git')
        os.chdir('fortune500')
ipy.system('pip install -r requirements.txt')


## 1. Setup & Google Cloud Authentication

In [ ]:
# Using GCP_PROJECT from configuration above

if GCP_PROJECT != "YOUR_PROJECT_ID":
    from IPython import get_ipython
    ipy = get_ipython()
    ipy.system(f"gcloud config set project {GCP_PROJECT}")
    ipy.system("gcloud auth login")
    result = __import__("subprocess").run(
        ["gcloud", "config", "get-value", "project"],
        capture_output=True, text=True
    )
    current = (result.stdout or "").strip() or "(unset)"
    print(f"✓ Project verified: {current}") if current == GCP_PROJECT else print(f"⚠ Current: {current}")
else:
    raise ValueError("Set GCP_PROJECT in the Configuration cell at the beginning of the notebook.")

In [ ]:
import subprocess
import json
import os

def check_billing(project_id):
    print(f"Checking billing status for {project_id}...")
    try:
        # Using gcloud to verify if billing is enabled on the given project
        result = subprocess.run(
            ["gcloud", "billing", "projects", "describe", project_id, "--format=json"], 
            capture_output=True, text=True, check=True
        )
        billing_info = json.loads(result.stdout)
        if billing_info.get("billingEnabled"):
            print("\u2705 Billing is enabled! You are ready to use BigQuery AI.")
        else:
            print("\u274c ERROR: Billing is NOT enabled for this project.")
            print("BigQuery AI will not function until billing is enabled in the Google Cloud Console.")
            print(f"Visit: https://console.cloud.google.com/billing/enable?project={project_id}")
            raise Exception("Billing not enabled")
    except subprocess.CalledProcessError as e:
        print("\u26a0 Could not verify billing status automatically. Output:")
        print(e.stderr)
        print("\nEnsure you are authenticated and have permissions (roles/billing.projectManager).")

if GCP_PROJECT and GCP_PROJECT != "YOUR_PROJECT_ID":
    check_billing(GCP_PROJECT)


### Ensure we're in the project root (where 00.0_scraper.py, SQL files, list.csv live)


In [ ]:
import os

def find_project_root():
    cwd = os.getcwd()
    for _ in range(5):
        if os.path.exists(os.path.join(cwd, "00.0_scraper.py")):
            return cwd
        parent = os.path.dirname(cwd)
        if parent == cwd:
            break
        cwd = parent
    return os.getcwd()

root = find_project_root()
os.chdir(root)
print(f"Working directory: {os.getcwd()}")

### Verify GCS bucket exists and is accessible


In [ ]:
import subprocess

result = subprocess.run(
    ["gsutil", "ls", GCS_BUCKET],
    capture_output=True,
    text=True,
)
if result.returncode == 0:
    print(f"✓ Bucket accessible: {GCS_BUCKET}")
else:
    print(f"Bucket not found. Creating {GCS_BUCKET}...")
    create = subprocess.run(["gsutil", "mb", "-l", BQ_LOCATION, GCS_BUCKET], capture_output=True, text=True)
    if create.returncode == 0:
        print(f"✓ Bucket created: {GCS_BUCKET}")
    else:
        print(f"⚠ Failed to create bucket: {create.stderr or create.stdout}")

## 2. Run Scraper (00.0_scraper.py)

In [ ]:
import asyncio
import sys

# Add project root to path
sys.path.insert(0, os.getcwd())

from importlib.util import spec_from_file_location, module_from_spec

# Load scraper module
spec = spec_from_file_location("scraper", "00.0_scraper.py")
scraper_mod = module_from_spec(spec)
spec.loader.exec_module(scraper_mod)

scraper = scraper_mod.SECScraper(
    limit=SCRAPER_LIMIT,
    last_n_years=SCRAPER_LAST_N_YEARS,
    workers=5,
    output_dir=SCRAPER_OUTPUT,
    dry_run=False,
)

# Scraper has internal tqdm; get expected count for context
import pandas as pd
try:
    n_companies = min(SCRAPER_LIMIT, len(pd.read_csv("list.csv")))
    print(f"Scraping up to {n_companies} companies ({SCRAPER_LAST_N_YEARS} years each)...")
except FileNotFoundError:
    n_companies = SCRAPER_LIMIT
await scraper.run()  # Jupyter has its own event loop; use await instead of asyncio.run()

## 3. Run Parser (00.1_parser.py) (Optional)

In [ ]:
import subprocess
import sys
from tqdm.auto import tqdm

# Count SGML filings to process
n_sgml = sum(1 for r, d, f in os.walk(SGML_DIR) if "full-submission.txt" in f)
with tqdm(total=max(1, n_sgml), desc="Parsing SGML to Markdown", unit="filing") as pbar:
    subprocess.run(
        [sys.executable, "00.1_parser.py", "--input_base", SGML_DIR, "--output_base", MARKDOWN_DIR],
        check=True,
        cwd=os.getcwd(),
    )
    pbar.update(max(1, n_sgml))

## 4. Extract Sections (01_extract_sections.py)

In [ ]:
import subprocess
import sys
from tqdm.auto import tqdm

# Count SGML filings to process
n_sgml = sum(1 for r, d, f in os.walk(SGML_DIR) if "full-submission.txt" in f)
with tqdm(total=max(1, n_sgml), desc="Extracting sections to JSONL", unit="filing") as pbar:
    subprocess.run(
        [sys.executable, "01_extract_sections.py", "--input_base", SGML_DIR, "--output_base", JSON_DIR],
        check=True,
        cwd=os.getcwd(),
    )
    pbar.update(max(1, n_sgml))

## 5. Upload JSONL to GCS

In [ ]:
import os
import glob
from IPython import get_ipython
from tqdm.auto import tqdm
json_src = os.path.abspath(JSON_DIR)
os.makedirs(json_src, exist_ok=True)
json_files = glob.glob(os.path.join(json_src, "*", "*", "sections.jsonl"))
n_json = len(json_files)
if n_json == 0:
    print(f"⚠ {json_src} is empty. Run Steps 2–4 (scraper, parser, extract sections) first, then re-run this cell.")
else:
    with tqdm(total=n_json, desc="Uploading JSONL to GCS", unit="file") as pbar:
        get_ipython().system(f'gsutil -m rsync -r {json_src}/ {GCS_BUCKET}/json')
        pbar.update(n_json)
    print(f"✓ Synced {n_json} section files to {GCS_BUCKET}/json")

## 6. BigQuery Pipeline

### Util Functions

In [ ]:
import subprocess
import glob
import json
from tqdm.auto import tqdm

SCHEMA = "filing_id:STRING,company:STRING,company_name:STRING,cik:STRING,sic:STRING,irs_number:STRING,state_of_inc:STRING,org_name:STRING,sec_file_number:STRING,film_number:STRING,business_street_1:STRING,business_street_2:STRING,business_city:STRING,business_state:STRING,business_zip:STRING,business_phone:STRING,mail_street_1:STRING,mail_street_2:STRING,mail_city:STRING,mail_state:STRING,mail_zip:STRING,filing_url:STRING,year:INTEGER,section_id:STRING,content:STRING"


def get_bq_distinct_companies() -> set[str]:
    sql = "SELECT DISTINCT company FROM `sec_filings.sections` WHERE company IS NOT NULL"
    cmd = [
        "bq",
        "query",
        "--use_legacy_sql=false",
        f"--location={BQ_LOCATION}",
        "--format=json",
        "--quiet",
        sql,
    ]
    bq_proj = globals().get("BQ_PROJECT", "")
    if bq_proj and bq_proj != "YOUR_PROJECT_ID":
        cmd.insert(-1, f"--project_id={bq_proj}")

    bq_env = os.environ.copy()
    bq_env.pop("PYTHONPATH", None)
    # Use system python to bypass local Conda/editable site-packages conflicts
    import os, sys
    if sys.platform == "darwin" and os.path.exists("/opt/homebrew/bin/python3"):
        bq_env["CLOUDSDK_PYTHON"] = "/opt/homebrew/bin/python3"
    elif sys.platform == "linux" and os.path.exists("/usr/bin/python3"):
        bq_env["CLOUDSDK_PYTHON"] = "/usr/bin/python3"
    elif sys.platform == "win32":
        pass # Let standard bq execution handle it on Windows, omit overriding
    import os
    if os.path.exists("/opt/homebrew/bin/python3"):
        bq_env["CLOUDSDK_PYTHON"] = "/opt/homebrew/bin/python3"
    elif os.path.exists("/usr/bin/python3"):
        bq_env["CLOUDSDK_PYTHON"] = "/usr/bin/python3"
    result = subprocess.run(cmd, capture_output=True, text=True, env=bq_env)
    if result.returncode != 0:
        return set()

    try:
        rows = json.loads(result.stdout or "[]")
    except json.JSONDecodeError:
        return set()

    done: set[str] = set()
    for r in rows:
        v = r.get("company") if isinstance(r, dict) else None
        if isinstance(v, str) and v.strip():
            done.add(v.strip())
    return done


def run_bq_query(sql_file: str):
    sql_path = os.path.join(os.getcwd(), sql_file)
    if not os.path.exists(sql_path):
        for d in [os.getcwd()] + [os.path.dirname(os.getcwd())] * 3:
            p = os.path.join(d, sql_file)
            if os.path.exists(p):
                sql_path = p
                break
    with open(sql_path) as f:
        sql = f.read()
    model = globals().get("GEMINI_MODEL")
    if model:
        import re
        sql = re.sub(r"OPTIONS \\(ENDPOINT = '[^']+'\\)", f"OPTIONS (ENDPOINT = '{model}')", sql)
    cmd = ["bq", "query", "--use_legacy_sql=false", f"--location={BQ_LOCATION}"]
    proj = globals().get("BQ_PROJECT", "YOUR_PROJECT_ID")
    if proj and proj != "YOUR_PROJECT_ID":
        cmd.extend([f"--project_id={proj}"])
    # Isolation fix for bq tool namespace conflicts
    bq_env = os.environ.copy()
    bq_env.pop("PYTHONPATH", None)
    # Use system python to bypass local Conda/editable site-packages conflicts
    import os, sys
    if sys.platform == "darwin" and os.path.exists("/opt/homebrew/bin/python3"):
        bq_env["CLOUDSDK_PYTHON"] = "/opt/homebrew/bin/python3"
    elif sys.platform == "linux" and os.path.exists("/usr/bin/python3"):
        bq_env["CLOUDSDK_PYTHON"] = "/usr/bin/python3"
    elif sys.platform == "win32":
        pass # Let standard bq execution handle it on Windows, omit overriding
    import os
    if os.path.exists("/opt/homebrew/bin/python3"):
        bq_env["CLOUDSDK_PYTHON"] = "/opt/homebrew/bin/python3"
    elif os.path.exists("/usr/bin/python3"):
        bq_env["CLOUDSDK_PYTHON"] = "/usr/bin/python3"
    try:
        subprocess.run(cmd, input=sql, check=True, env=bq_env, capture_output=True, text=True)
    except subprocess.CalledProcessError as e:
        print(f"bq error:\n{e.stderr}")
        raise


def run_bq_load(uris: str):
    cmd = ["bq", "load", "--source_format=NEWLINE_DELIMITED_JSON", "--replace", f"--schema={SCHEMA}", "sec_filings.sections_staging", uris]
    bq_proj = globals().get("BQ_PROJECT", "")
    if bq_proj and bq_proj != "YOUR_PROJECT_ID":
        cmd.insert(2, f"--project_id={bq_proj}")
    bq_env = os.environ.copy()
    bq_env.pop("PYTHONPATH", None)
    # Use system python to bypass local Conda/editable site-packages conflicts
    import os, sys
    if sys.platform == "darwin" and os.path.exists("/opt/homebrew/bin/python3"):
        bq_env["CLOUDSDK_PYTHON"] = "/opt/homebrew/bin/python3"
    elif sys.platform == "linux" and os.path.exists("/usr/bin/python3"):
        bq_env["CLOUDSDK_PYTHON"] = "/usr/bin/python3"
    elif sys.platform == "win32":
        pass # Let standard bq execution handle it on Windows, omit overriding
    import os
    if os.path.exists("/opt/homebrew/bin/python3"):
        bq_env["CLOUDSDK_PYTHON"] = "/opt/homebrew/bin/python3"
    elif os.path.exists("/usr/bin/python3"):
        bq_env["CLOUDSDK_PYTHON"] = "/usr/bin/python3"
    subprocess.run(cmd, check=True, capture_output=True, env=bq_env)


def process_batch(uris: str):
    run_bq_load(uris)
    run_bq_query("04_extraction.sql")
    insert_cmd = ["bq", "query", "--use_legacy_sql=false", f"--location={BQ_LOCATION}",
         "INSERT INTO sec_filings.sections SELECT * FROM sec_filings.sections_staging"]
    bq_proj = globals().get("BQ_PROJECT", "")
    if bq_proj and bq_proj != "YOUR_PROJECT_ID":
        insert_cmd.insert(-1, f"--project_id={bq_proj}")
    bq_env = os.environ.copy()
    bq_env.pop("PYTHONPATH", None)
    # Use system python to bypass local Conda/editable site-packages conflicts
    import os, sys
    if sys.platform == "darwin" and os.path.exists("/opt/homebrew/bin/python3"):
        bq_env["CLOUDSDK_PYTHON"] = "/opt/homebrew/bin/python3"
    elif sys.platform == "linux" and os.path.exists("/usr/bin/python3"):
        bq_env["CLOUDSDK_PYTHON"] = "/usr/bin/python3"
    elif sys.platform == "win32":
        pass # Let standard bq execution handle it on Windows, omit overriding
    import os
    if os.path.exists("/opt/homebrew/bin/python3"):
        bq_env["CLOUDSDK_PYTHON"] = "/opt/homebrew/bin/python3"
    elif os.path.exists("/usr/bin/python3"):
        bq_env["CLOUDSDK_PYTHON"] = "/usr/bin/python3"
    subprocess.run(insert_cmd, check=True, env=bq_env)


### 6.0 Initialize tables


In [ ]:
sql_path = os.path.join(os.getcwd(), "03_init_tables.sql")
n_tables = 0
if os.path.exists(sql_path):
    with open(sql_path) as f:
        n_tables = sum(1 for line in f if "CREATE" in line and "TABLE" in line) or 1
with tqdm(total=max(1, n_tables), desc="Initializing BigQuery tables", unit="table") as pbar:
    run_bq_query("03_init_tables.sql")
    pbar.update(max(1, n_tables))
print("Tables initialized.")

### 6.1 Batch load, extract, archive


In [ ]:
import subprocess
from tqdm.auto import tqdm

START_OVER = False  # Set True to re-run everything (run 6.0 Init Tables first to avoid duplicates)

company_dirs_all = sorted(glob.glob(os.path.join(JSON_DIR, "*")))
company_dirs_all = [d for d in company_dirs_all if os.path.isdir(d)]

already_done = set() if START_OVER else get_bq_distinct_companies()

company_dirs = company_dirs_all
if already_done and not START_OVER:
    n_total = len(company_dirs_all)
    n_done = sum(1 for d in company_dirs_all if os.path.basename(d) in already_done)
    pct_done = (n_done / n_total * 100) if n_total else 0.0

    ans = input(
        f"Found {n_done}/{n_total} companies already in BigQuery ({pct_done:.1f}%). Resume from where it stopped? [Y/n] "
    ).strip().lower()
    if ans in ("n", "no"):
        print(
            "Not resuming. To start over, set START_OVER=True above (and run 6.0 Init Tables first to avoid duplicates)."
        )
        RUN_BATCH = False
    else:
        RUN_BATCH = True
        company_dirs = [d for d in company_dirs_all if os.path.basename(d) not in already_done]
        print(f"Resuming: processing {len(company_dirs)}/{n_total} remaining companies.")
else:
    RUN_BATCH = True
    if START_OVER:
        print(
            "START_OVER=True: processing all companies (run 6.0 Init Tables first to avoid duplicates)."
        )

all_sections = []
for d in company_dirs:
    all_sections.extend(glob.glob(os.path.join(d, "*", "sections.jsonl")))
n_section_files = len(all_sections)
n_companies = len(company_dirs)

gcs_prefix = f"{GCS_BUCKET}/json"

if RUN_BATCH:
    for idx, company_dir in enumerate(
        tqdm(
            company_dirs,
            desc=f"Processing {n_companies} companies ({n_section_files} section files)",
            unit="company",
        )
    ):
        sections = glob.glob(os.path.join(company_dir, "*", "sections.jsonl"))
        if not sections:
            continue
        uris = [
            f"{gcs_prefix}/" + os.path.relpath(p, JSON_DIR).replace(os.sep, "/")
            for p in sections
        ]
        uris_str = ",".join(uris)
        company_ticker = os.path.basename(company_dir)
        pct = (idx + 1) / n_companies * 100
        try:
            process_batch(uris_str)
        except subprocess.CalledProcessError as e:
            err_detail = (
                (e.stderr or e.stdout or "").strip() if isinstance((e.stderr or e.stdout), str) else (e.stderr or e.stdout or b"").decode(errors="replace").strip()
                if (e.stderr or e.stdout)
                else str(e)
            )
            print(f"\n⚠ [{company_ticker}] FAILED at {pct:.1f}% ({idx + 1}/{n_companies}): {e}")
            if err_detail:
                print(f"   {err_detail[:500]}")
            continue
        except Exception as e:
            print(
                f"\n⚠ [{company_ticker}] FAILED at {pct:.1f}% ({idx + 1}/{n_companies}): {type(e).__name__}: {e}"
            )
            continue

### 6.2 Create graph edges


In [ ]:
with tqdm(total=1, desc="Creating graph edges", unit="step") as pbar:
    run_bq_query("05_create_graph.sql")
    pbar.update(1)
print("Graph edges created.")

### 6.3 Prepare node/edge tables


In [ ]:
with tqdm(total=1, desc="Preparing node/edge tables", unit="step") as pbar:
    run_bq_query("06_prepare_property_graph.sql")
    pbar.update(1)
print("Property graph tables created.")

### 6.4 Add market_action to nodes_market


In [ ]:
# Note: If market_action column already exists, ALTER TABLE will fail on re-run.
# Edit 06_1_add_action_to_market.sql to remove the ADD COLUMN line in that case.
with tqdm(total=1, desc="Adding market_action to nodes_market", unit="step") as pbar:
    run_bq_query("06_1_add_action_to_market.sql")
    pbar.update(1)
print("Market action column added.")

### 6.5 Create SecGraph property graph


In [ ]:
with tqdm(total=1, desc="Creating SecGraph property graph", unit="step") as pbar:
    run_bq_query("07_create_property_graph_ddl.sql")
    pbar.update(1)
print("Pipeline complete.")